In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import tensorflow as tf
import IPython
import warnings
warnings.filterwarnings("ignore")

In [2]:
y_test = np.random.randint(0,2,(3,4))
y_pred = np.random.random((3,4))
y_test,y_pred

(array([[1, 1, 0, 0],
        [0, 0, 0, 1],
        [0, 0, 1, 1]]),
 array([[0.88769816, 0.34427635, 0.80663313, 0.23713164],
        [0.92247084, 0.32539117, 0.56029389, 0.19014539],
        [0.55936822, 0.2639554 , 0.18653309, 0.48330444]]))

In [3]:
def log_loss(y_test,y_pred):
    y_test = y_test.astype(np.float16)
    y_pred = y_pred.astype(np.float16)
    N,M = y_test.shape
    a=[]
    for m in range(M):
        loss=0
        for i in range(N):
            loss -= ((y_test[i,m]*np.log(y_pred[i,m]))+((1.0-y_test[i,m])*np.log(1.0-y_pred[i,m])))
        loss = loss/N
        a.append(round(loss,8))
    return a
a = log_loss(y_test,y_pred)
print(a)
np.mean(a)

[1.16495329, 0.58883705, 1.38100981, 0.88598122]


1.0051953425

In [4]:
tf.keras.losses.categorical_crossentropy(np.transpose(y_test), np.transpose(y_pred)).numpy()

array([0.98181818, 0.99762795, 2.11963138, 2.19973099])

In [5]:
print(tf.keras.losses.binary_crossentropy(np.transpose(y_test), np.transpose(y_pred)).numpy())
tf.keras.losses.binary_crossentropy(np.transpose(y_test), np.transpose(y_pred)).numpy().mean()

[1.16525627 0.58879895 1.38132001 0.88591458]


1.005322451436327

# how to improve log loss

# 1. log loss is nothing but entropy.
# 2. entropy is measure of uncertainty.
# 3. so a low Log Loss means alow uncertainty/entropy of your model

# The use of the logarithm provides extreme punishments for being both confident and wrong. In the worst possible case, a prediction that something is true when it is actually false will add an infinite amount to your error score, it would be much better to keep our probabilities between 0.05–0.95 so that we are never very sure about our prediction. In this case, we won’t see the massive growth of an error function.

In [6]:
y_test = np.random.randint(0,2,(300,1))
y_pred = np.random.random((300,1))
a = log_loss(y_test,y_pred)
print(a)

[1.00184229]


In [7]:
y_pred[0.1>y_pred].shape,y_pred[y_pred>0.9].shape 

((25,), (27,))

In [8]:
y = y_pred.clip(0.05,0.95)
log_loss(y_test,y)

[0.9647188]

In [9]:
y_test = np.random.randint(0,2,(300,3))
y_pred = np.random.random((300,3))
a = log_loss(y_test,y_pred)
print(a)

[0.96954779, 1.03297325, 1.02578139]


In [10]:
y = y_pred.clip(0.05,0.95)
log_loss(y_test,y)

[0.92603719, 0.98996026, 0.96475194]

# The Trick is:
# We need to change all values that less than 0.05 to be equal to 0.05 and all values that are more than 0.95 to be equal to 0.95

# secound Trick is (note : if only if you have 100 % accuracy)

In [11]:
a = np.array([1.0,0.0,1.0,0.0,1.0])
p = np.array([0.6,0.3,0.7,0.2,0.9])

In [12]:
tf.keras.losses.binary_crossentropy(a, p).numpy()

0.31053577802469534

In [13]:
tf.keras.losses.binary_crossentropy(a, np.where(p,p>=0.5,1).astype(np.float16)).numpy()

0.0

# what if you have 80 % accuracy 

In [14]:
a = np.array([1.0,0.0,1.0,0.0,1.0])
p = np.array([0.6,0.3,0.7,0.2,0.4])

In [15]:
tf.keras.losses.binary_crossentropy(a, p).numpy()

0.47272179349018834

In [16]:
tf.keras.losses.binary_crossentropy(a, np.where(p,p>=0.5,1).astype(np.float16)).numpy()

3.05

# Log loss has very useful property in that it penalizes heavily when the model makes emphatically incorrect labels. In case the model predicts 1 for the label but the truth is 0, you will end up with natural log of (0), which reaches negative infinity. This will greatly increase the log loss term.

# By this better to not to use secound trick. if you use hahaha you know 0.47 is for better than 3.05 :D

# Hope this helps with the intuition.